# Machine Learning - Late Delivery Risk Prediction

This notebook develops a supervised machine-learning model to predict late delivery risk using the cleaned DataCo Supply Chain dataset.

The target variable is `late_delivery_risk`, which indicates whether an order is at risk of late delivery.

The machine-learning process will include:

1. Loading the cleaned dataset
2. Selecting the target and input features
3. Preparing the data
4. Splitting the data into training and testing sets
5. Training a classification model
6. Making predictions
7. Evaluating the model
8. Interpreting the results

In [3]:
!pip install scikit-learn

  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.2 MB 8.3 MB/s eta 0:00:01
   --------------- ------------------------ 3.1/8.2 MB 10.8 MB/s eta 0:00:01
   ---------------------------- ----------- 5.8/8.2 MB 11.0 MB/s eta 0:00:01
   ---------------------------------------  8.1/8.2 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 10.6 MB/s eta 0:00:00
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Import Libraries

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Define the Target Variable

The aim of this project is to predict whether an order is at risk of late delivery.

The target variable is `late_delivery_risk`, where:

- `0` = Not at risk of late delivery
- `1` = At risk of late delivery

In [5]:
target = "late_delivery_risk"

print(df[target].value_counts())

late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64


## Select Features

The model needs input variables, known as features, to make predictions about late delivery risk.

I selected features that are relevant to shipping performance, order characteristics, customer information and timing.

I have not included `shipping_delay` or `late_delivery_flag` because these variables are directly calculated from delivery information and could give the model information about the target variable. Excluding them helps avoid data leakage.

In [6]:
features = [
    "days_for_shipping_real",
    "days_for_shipment_scheduled",
    "benefit_per_order",
    "sales_per_customer",
    "category_name",
    "customer_segment",
    "shipping_mode",
    "order_item_total",
    "order_item_quantity",
    "order_month",
    "order_weekday"
]

X = df[features]
y = df[target]

print("Features:", X.columns.tolist())
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['days_for_shipping_real', 'days_for_shipment_scheduled', 'benefit_per_order', 'sales_per_customer', 'category_name', 'customer_segment', 'shipping_mode', 'order_item_total', 'order_item_quantity', 'order_month', 'order_weekday']
X shape: (180519, 11)
y shape: (180519,)


## Prepare Categorical Features

Some of the selected features contain categorical values rather than numbers.

These categorical features need to be converted into numerical values before they can be used by the machine-learning model.

One-hot encoding will be used so that the categories can be represented as numerical columns without creating an incorrect order between the categories.

In [7]:
X = pd.get_dummies(X, drop_first=True)

print("Shape after encoding:", X.shape)
X.head()

Shape after encoding: (180519, 67)


,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,order_item_total,order_item_quantity,order_month,category_name_As Seen on TV!,category_name_Baby,category_name_Baseball & Softball,...,customer_segment_Home Office,shipping_mode_Same Day,shipping_mode_Second Class,shipping_mode_Standard Class,order_weekday_Monday,order_weekday_Saturday,order_weekday_Sunday,order_weekday_Thursday,order_weekday_Tuesday,order_weekday_Wednesday
0,3,4,91.250000,314.640015,314.640015,1,1,False,False,False,...,False,False,False,True,False,False,False,False,False,True
1,5,4,-249.089996,311.359985,311.359985,1,1,False,False,False,...,False,False,False,True,False,True,False,False,False,False
2,4,4,-247.779999,309.720001,309.720001,1,1,False,False,False,...,False,False,False,True,False,True,False,False,False,False
3,3,4,22.860001,304.809998,304.809998,1,1,False,False,False,...,True,False,False,True,False,True,False,False,False,False
4,2,4,134.210007,298.250000,298.250000,1,1,False,False,False,...,False,False,False,True,False,True,False,False,False,False


## Split the Data into Training and Testing Sets

The dataset will be divided into training and testing data.

80% of the data will be used to train the model, while 20% will be kept separate for testing.

The test data allows the model to be evaluated on data that it has not seen during training.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training features: (144415, 67)
Testing features: (36104, 67)
Training target: (144415,)
Testing target: (36104,)


## Train the Machine Learning Model

A Decision Tree Classifier will be used to predict whether an order is at risk of late delivery.

A decision tree makes predictions by splitting the data based on different features. It is suitable for this project because the target variable has two possible outcomes: late delivery risk or no late delivery risk.

The model will be trained using the training dataset.

In [9]:
model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

model.fit(X_train, y_train)

print("Model training completed.")

Model training completed.


## Make Predictions

The trained model will now be used to predict late delivery risk for the testing dataset.

These predictions will then be compared with the actual results to evaluate how well the model performs.

In [10]:
y_pred = model.predict(X_test)

print("Predictions completed.")
print("Number of predictions:", len(y_pred))

Predictions completed.
Number of predictions: 36104


## Evaluate Model Accuracy

Accuracy measures the percentage of predictions that the model got correct.

It provides a simple first measure of how well the Decision Tree performed on the unseen testing data.

In [11]:
accuracy = accuracy_score(y_test, y_pred)

print(f"Model accuracy: {accuracy:.2%}")

Model accuracy: 97.45%


## Confusion Matrix

A confusion matrix shows the number of correct and incorrect predictions for each class.

It allows us to see how well the model predicts both:

- `0` = Not at risk of late delivery
- `1` = At risk of late delivery

This is useful because accuracy alone does not show how the model performs for each class.

In [12]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[15386   922]
 [    0 19796]]


15,386 - correctly predicted not late,
922 - predicted late when it wasn't,
0 - missed late deliveries,
19,796 - correctly predicted late

### Model Evaluation and Data Leakage

The model achieved an accuracy of 97.45%. However, the confusion matrix shows that it correctly identified all 19,796 orders at risk of late delivery in the testing dataset and missed 0.

The testing dataset contains 36,104 orders, so these figures only represent the test data and not the full dataset.

The very high accuracy and the fact that no late-risk orders were missed suggests that some of the selected features may be giving the model information that is too closely related to the target variable.

The features `days_for_shipping_real` and `days_for_shipment_scheduled` are directly related to whether an order is considered late. Using these features could therefore cause **data leakage**, where the model has access to information that would not realistically be available when making an early prediction.

For this reason, these features will be removed and the model will be retrained using information that would be available before the delivery outcome is known.

## Remove Features Causing Data Leakage

The previous model used actual and scheduled shipping times, which are closely related to the late delivery outcome.

To make the prediction more realistic, these features will be removed from the model.

The remaining features describe the order, product, customer and shipping information that could be available before the delivery outcome is known.

In [13]:
features_clean = [
    "benefit_per_order",
    "sales_per_customer",
    "category_name",
    "customer_segment",
    "shipping_mode",
    "order_item_total",
    "order_item_quantity",
    "order_month",
    "order_weekday"
]

X_clean = df[features_clean]

X_clean = pd.get_dummies(X_clean, drop_first=True)

y = df["late_delivery_risk"]

print("Features after removing leakage:", X_clean.shape)

Features after removing leakage: (180519, 65)


## Split the Cleaned Data

The cleaned features will now be divided into training and testing data.

80% will be used to train the model and 20% will be used to test its performance on unseen data.

In [14]:
X_train_clean, X_test_clean, y_train_clean, y_test_clean = train_test_split(
    X_clean,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training features:", X_train_clean.shape)
print("Testing features:", X_test_clean.shape)
print("Training target:", y_train_clean.shape)
print("Testing target:", y_test_clean.shape)

Training features: (144415, 65)
Testing features: (36104, 65)
Training target: (144415,)
Testing target: (36104,)


## Train the Improved Decision Tree Model

The Decision Tree will now be retrained using the cleaned feature set.

The features that could cause data leakage have been removed, so this model should provide a more realistic measure of how well late delivery risk can be predicted.

In [15]:
model_clean = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

model_clean.fit(X_train_clean, y_train_clean)

print("Improved model training completed.")

Improved model training completed.


## Make Predictions with the Improved Model

The improved model will now be used to predict late delivery risk for the testing data.

The predictions will be compared with the actual results to evaluate the model's performance.

In [16]:
y_pred_clean = model_clean.predict(X_test_clean)

print("Predictions completed.")
print("Number of predictions:", len(y_pred_clean))

Predictions completed.
Number of predictions: 36104


## Evaluate the Improved Model

The accuracy of the improved model will be calculated using the unseen testing data.

This will allow the performance of the model to be compared with the original model that contained potential data leakage.

In [17]:
accuracy_clean = accuracy_score(y_test_clean, y_pred_clean)

print(f"Improved model accuracy: {accuracy_clean:.2%}")

Improved model accuracy: 69.63%


### Initial Evaluation

The improved model achieved an accuracy of 69.63% on the testing data.

This is lower than the original model's accuracy of 97.45%. However, the original model included features that were closely related to the target variable and could cause data leakage.

After removing these features, the lower accuracy provides a more realistic indication of the model's ability to predict late delivery risk.

Further evaluation will be carried out using a confusion matrix and classification report to understand how well the model predicts each class.

## Confusion Matrix for the Improved Model

The confusion matrix shows how many orders were correctly and incorrectly classified by the improved model.

This helps us understand whether the model is better at identifying orders at risk of late delivery or orders that are not at risk.

In [18]:
cm_clean = confusion_matrix(y_test_clean, y_pred_clean)

print("Confusion Matrix:")
print(cm_clean)

Confusion Matrix:
[[14374  1934]
 [ 9030 10766]]


14,374 - correctly predicted not at risk,
1,934 - predicted at risk, but actually not,
9,030 - predicted not at risk, but actually at risk,
10,766 - correctly predicted at risk

### Confusion Matrix Interpretation

The confusion matrix shows that the model correctly predicted 14,374 orders that were not at risk of late delivery and 10,766 orders that were at risk.

However, the model incorrectly classified 9,030 orders that were actually at risk as not at risk.

This shows that although the model has a reasonable overall accuracy, it does not identify all late-risk orders successfully. This is important because missing an order that is at risk of late delivery could be a significant issue in a real supply-chain situation.

## Classification Report

The classification report provides additional measures to evaluate the model.

- **Precision** shows how often the model's positive predictions were correct.
- **Recall** shows how many of the actual positive cases the model successfully identified.
- **F1-score** combines precision and recall into one measure.

These measures help provide a more complete evaluation of the model than accuracy alone.

In [19]:
print(classification_report(
    y_test_clean,
    y_pred_clean,
    target_names=["Not at Risk", "At Risk"]
))

              precision    recall  f1-score   support

 Not at Risk       0.61      0.88      0.72     16308
     At Risk       0.85      0.54      0.66     19796

    accuracy                           0.70     36104
   macro avg       0.73      0.71      0.69     36104
weighted avg       0.74      0.70      0.69     36104



### Classification Report Interpretation

The classification report shows that the model performs differently for the two classes.

For orders that are not at risk, the model has a precision of 61% and a recall of 88%. This means the model identifies most of the orders that are not at risk, but some of its predictions for this class are incorrect.

For orders that are at risk, the model has a precision of 85% and a recall of 54%. The high precision means that when the model predicts an order is at risk, it is correct 85% of the time. However, the lower recall means that the model only identifies 54% of the orders that are actually at risk.

Overall, the model achieved an accuracy of 69.63%. The model is therefore quite reliable when it predicts an order is at risk, but it misses a significant number of actual at-risk orders.

This means the model could be useful as a starting point for identifying delivery risks, but further improvements would be needed before using it for real-world decisions.

## Feature Importance

Feature importance shows which features had the greatest influence on the Decision Tree's predictions.

This helps explain what information the model is using when predicting late delivery risk.

In [20]:
feature_importance = pd.DataFrame({
    "feature": X_clean.columns,
    "importance": model_clean.feature_importances_
})

feature_importance = feature_importance.sort_values(
    "importance",
    ascending=False
)

feature_importance.head(10)

,feature,importance
58,shipping_mode_Standard Class,0.798069
56,shipping_mode_Same Day,0.136692
57,shipping_mode_Second Class,0.058181
0,benefit_per_order,0.001801
2,order_item_total,0.000844
62,order_weekday_Thursday,0.000801
4,order_month,0.000501
55,customer_segment_Home Office,0.000475
59,order_weekday_Monday,0.000465
64,order_weekday_Wednesday,0.000452


In [22]:
import plotly.express as px

In [23]:
fig = px.bar(
    feature_importance.head(10),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 10 Features Used by the Decision Tree",
    labels={
        "importance": "Feature Importance",
        "feature": "Feature"
    }
)

fig.update_layout(yaxis={"categoryorder": "total ascending"})

fig.show()

### Feature Importance Interpretation

The feature importance results show that shipping mode is the most important factor used by the Decision Tree when predicting late delivery risk.

Standard Class has the highest importance, followed by Same Day and Second Class. Other features have much smaller importance values.

This suggests that shipping mode provides useful information for predicting late delivery risk. This also supports the findings from the EDA, where different shipping modes showed different late delivery rates.

However, feature importance shows which features the model uses for prediction. It does not mean that shipping mode directly causes late deliveries.

## Machine Learning Conclusion

A Decision Tree Classifier was used to predict whether an order was at risk of late delivery.

The first model achieved 97.45% accuracy, but further investigation showed that some features could cause data leakage. These features were removed and the model was retrained.

The improved model achieved an accuracy of 69.63%. It had a precision of 85% for orders at risk of late delivery, meaning that most orders predicted as at risk were correctly identified. However, its recall for the At Risk class was 54%, meaning that the model missed a number of orders that were actually at risk.

The feature importance results showed that shipping mode was the most important feature used by the model. This was consistent with the EDA findings, where late delivery rates varied between shipping modes.

Overall, the model provides a useful starting point for predicting late delivery risk, but it would need further improvement before being used for real-world decision-making. Future improvements could include testing other machine learning algorithms and tuning the model's parameters.